# Automated Document Analysis and OCR System for Multi Format Documents

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import morphology, filters
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from scipy.ndimage import interpolation
import pytesseract
from PIL import Image
import json

## Foundation and Preprocessing

### Image Loading and Conversion

In [2]:
class ImageLoader:
    def __init__(self, image_path):
        self.supported_formats = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.pdf')
    
    def load_image(self, image_path):
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Unsupported image format or unable to load image: {image_path}")
        return image
    
    def convert_to_grayscale(self, image):
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        return gray
    
    def resize_image(self, image, max_width=1500):
        height, width = image.shape[:2]
        if width > max_width:
            scale = max_width / float(width)
            new_height = int(height * scale)
            resized = cv2.resize(image, (max_width, new_height), interpolation=cv2.INTER_AREA)
            return resized
        return image
    

### Noice Reduction Implementation

In [ ]:
class NoiceReducer:
    def gaussuan_blur(self, image, kernel_size = 5, sigma = 1.0):
        blurred = cv2.GaussianBlur(image, (kernel_size, kernel_size), sigma)
        return blurred

    def bilateral_filter(self, image, diameter = 9, sigma_color = 75, sigma_space = 75):
        filtered = cv2.bilateralFilter(image, diameter, sigma_color, sigma_space)
        return filtered

    def morphological_noice_removal(self, binary_image):
        kernel = np.ones(2,2, np.uint8)
        opening = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel, iterations=1)
        kernel = np.ones(3,3, np.uint8)
        closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=1)
        return closing

    

### Binarization Techniques

In [ ]:
class Binarizer:
    def otsu_threshold(self, image):
        thresh_val, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary, thresh_val
    def adaptive_threshold(self, image, block_size = 11, C = 2):
        binary = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block_size, C)
        return binary
    def sauvola_threshold(self, image, k = 0.2, window_size = 15):
        from skimage.filters import threshold_sauvola
        thresh_sauvola = threshold_sauvola(image, window_size=window_size, k=k)
        binary = (image > thresh_sauvola).astype(np.uint8) * 255
        return binary

### Perspective Correction and Corner Detection

In [ ]:
class PerspectiveCorrector:
    def detect_corners(self, binary_image):
        corners = cv2.goodFeaturesToTrack(binary_image, maxCorners=4, qualityLevel=0.01, minDistance=30)
        if corners is None:
            return None
        
        return corners.reshape(-1, 2)

    def perspective_correction(self, image, corners):
        if corners is None or len(corners) != 4:
           return image
        
        corners = self._order_points(corners)
        width = int(
            max(
                np.linalg.norm(corners[0] - corners[1]),
                np.linalg.norm(corners[2] - corners[3])
            )
        )

        height = int(
            max(
                np.linalg.norm(corners[0] - corners[3]),
                np.linalg.norm(corners[1] - corners[2])
            )
        )

        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype="float32")
        M = cv2.getPerspectiveTransform(corners.astype("float32"), dst)
        corrected = cv2.warpPerspective(image, M, (width, height))
        return corrected

    def _order_points(self, pts):
        rect = np.zeros((4, 2), dtype="float32")
        s = pts.sum(axis=1)
        
        rect[0] = pts[np.argmin(s)]
        rect[2] = pts[np.argmax(s)]
        rect[1] = pts[np.argmin(np.diff(s))]
        rect[3] = pts[np.argmax(np.diff(s))]
        return rect